In [29]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    to_timestamp,
    col,
    window,
    avg,
    round as _round,
    count
)

In [30]:
spark = (
    SparkSession.builder
    .appName("Lab2-Transactions")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — gotowy")

df = spark.read.json("transactions_10k.jsonl")
df = df.withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))

print(f"Liczba rekordów: {df.count()}")
df.printSchema()

Spark 4.0.0-preview2 — gotowy
Liczba rekordów: 10000
root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [31]:
def avg_amount_by_window(df, store_name, window_duration):
    return (
        df.filter(col("store") == store_name)
        .groupBy(window("timestamp", window_duration))
        .agg(
            _round(avg("amount"), 2).alias("srednia")
        )
        .select(
            col("window.start").alias("od"),
            col("window.end").alias("do"),
            "srednia"
        )
        .orderBy("od")
        .limit(1)
    )

avg_amount_by_window(df, "Gdańsk", "1 hour").show(truncate=False)

+-------------------+-------------------+-------+
|od                 |do                 |srednia|
+-------------------+-------------------+-------+
|2026-04-12 08:00:00|2026-04-12 09:00:00|395.01 |
+-------------------+-------------------+-------+



In [32]:
def count_transactions_by_category(df, start_time, end_time):
    return (
        df.filter(
            (col("timestamp") >= start_time) &
            (col("timestamp") < end_time)
        )
        .groupBy("category")
        .agg(
            count("tx_id").alias("liczba_tx")
        )
        .orderBy("category")
    )

count_transactions_by_category(df,"2026-04-12 09:00:00","2026-04-12 09:30:00").show(truncate=False)

+-----------+---------+
|category   |liczba_tx|
+-----------+---------+
|elektronika|611      |
|książki    |622      |
|odzież     |605      |
|żywność    |567      |
+-----------+---------+



In [33]:
def peak_transactions_window(df, window_duration):
    return (
        df.groupBy(window("timestamp", window_duration))
        .agg(
            count("tx_id").alias("liczba_tx")
        )
        .select(
            col("window.start").alias("od"),
            col("window.end").alias("do"),
            "liczba_tx"
        )
        .orderBy(col("liczba_tx").desc())
        .limit(1)
    )
    
peak_transactions_window(df, "15 minutes").show(truncate=False)

+-------------------+-------------------+---------+
|od                 |do                 |liczba_tx|
+-------------------+-------------------+---------+
|2026-04-12 09:15:00|2026-04-12 09:30:00|1234     |
+-------------------+-------------------+---------+

